# Modulation

> Bits to symbols, and back.

In [ ]:
#| default_exp dsp.modulation

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from __future__ import annotations

from abc import ABC, abstractmethod

import numpy as np

from comm_core.block import _is_torch

In [ ]:
#| export
def _conj(h):
    return h.conj() if _is_torch(h) else np.conj(h)


def _real(z):
    if _is_torch(z):
        return z.real if z.is_complex() else z
    return np.real(z)

## Modulation

A `Modulation` maps bits to symbols with average energy 1. It detects them again knowing the channel coefficient
`h` (coherent detection, with perfect channel knowledge at the receiver).

`BPSK` is Sionna's 1-bit PAM, with Sionna's labelling: bit 0 → +1, bit 1 → −1. It is written here, on numpy
arrays or torch tensors, so that the core needs no Sionna. `SionnaModulation('pam', 1)` gives the same symbols
and the same decisions.

In [ ]:
#| export
class Modulation(ABC):
    "Bits <-> symbols."

    bits_per_symbol: int = 1

    @abstractmethod
    def modulate(self, bits):
        "Symbols (..., num_bits / bits_per_symbol), with average energy 1."

    @abstractmethod
    def demodulate(self, received, h):
        "The detected bits (booleans) of the received symbols, knowing the channel coefficients `h`."


class BPSK(Modulation):
    "Binary phase-shift keying, labelled as Sionna's 1-bit PAM: bit 0 -> +1, bit 1 -> -1."

    bits_per_symbol = 1

    def modulate(self, bits):
        return 1.0 - 2.0 * bits

    def demodulate(self, received, h):
        "Matched filter, which is the maximum-likelihood detector for BPSK: bit 1 where Re(y h*) is negative."
        return _real(received * _conj(h)) < 0

In [ ]:
show_doc(BPSK.demodulate)

In [ ]:
bpsk = BPSK()
b = np.array([0, 1, 1, 0])
assert bpsk.modulate(b).tolist() == [1, -1, -1, 1]                      # Sionna's labelling
h = np.exp(1j * 2.0)                                               # any phase: the receiver knows h
assert (bpsk.demodulate(h * bpsk.modulate(b), h) == b).all()

## Sionna's constellations

`SionnaModulation` makes any of Sionna's constellations a `Modulation`: QAM, PAM, or a custom one. It modulates
with Sionna's `Mapper`. It detects with a one-tap equalizer, `y / h`, then Sionna's `Demapper` with hard decisions
(max-log: the closest point's bits). So a `BitLevelLink` (see `link_level`) can send 16-QAM, or any other constellation,
uncoded:

```python
LinkLevelChannel(modulation=SionnaModulation('qam', 4), ...)
```

It takes and returns numpy arrays or torch tensors, like `BPSK`. It needs Sionna (PyTorch):
`pip install 'comm-core[sionna]'`. For coded 5G NR, with soft decisions (LLRs) and LDPC decoding, use `SionnaLink`.

In [ ]:
#| export
class SionnaModulation(Modulation):
    "A Sionna constellation (QAM, PAM, custom) as a `Modulation`: Sionna's `Mapper`; a one-tap equalizer, then its `Demapper`'s hard decisions."

    def __init__(
        self,
        constellation_type: str | None = 'qam', # Sionna's 'qam', 'pam' or 'custom'; None with `constellation`
        num_bits_per_symbol: int | None = 2, # Bits per symbol, e.g. 4 for 16-QAM; None with `constellation`
        constellation=None, # A `sionna.phy.mapping.Constellation`, instead of the two above
        device: str | None = None, # Torch device; None: Sionna's default (a GPU if there is one)
        precision: str = 'single', # 'single' or 'double'
    ):
        import torch
        from sionna.phy.mapping import Demapper, Mapper
        self._torch = torch
        if constellation is not None:
            constellation_type = num_bits_per_symbol = None
        self.mapper = Mapper(constellation_type, num_bits_per_symbol, constellation=constellation, precision=precision,
                             device=device)
        self.constellation = self.mapper.constellation
        self.demapper = Demapper('maxlog', constellation=self.constellation, hard_out=True, precision=precision,
                                 device=device)
        self.bits_per_symbol = int(self.constellation.num_bits_per_symbol)
        self.device = self.mapper.device
        self.real = torch.float64 if precision == 'double' else torch.float32
        self.complex = torch.complex128 if precision == 'double' else torch.complex64

    def modulate(self, bits):
        "Sionna's `Mapper`: (..., num_bits / bits_per_symbol) symbols."
        x = self.mapper(self._torch.as_tensor(bits, device=self.device).to(self.real))
        return x if _is_torch(bits) else x.cpu().numpy().astype(np.complex128)

    def demodulate(self, received, h):
        "The closest point's bits (Sionna's max-log `Demapper`, hard decisions), after the one-tap equalizer y / h."
        torch = self._torch
        y = torch.as_tensor(received, device=self.device).to(self.complex)
        hh = torch.as_tensor(h, device=self.device).to(self.complex)
        bits = self.demapper(y / hh, torch.ones((), dtype=self.real, device=self.device)) > 0.5
        return bits if _is_torch(received) else bits.cpu().numpy()

In [ ]:
show_doc(SionnaModulation.demodulate)

In [ ]:
from math import erfc, sqrt

import torch

from comm_core.dsp.signal import apply_channel

In [ ]:
#| sionna
# 1-bit PAM is BPSK: the same symbols, the same decisions
pam = SionnaModulation('pam', 1)
rng = np.random.default_rng(0)
b = rng.integers(0, 2, (3, 64))
assert np.allclose(pam.modulate(b), bpsk.modulate(b))
y = apply_channel(bpsk.modulate(b), np.exp(1j * 0.4), 1.0, 0.5, rng)
assert (pam.demodulate(y, np.exp(1j * 0.4)) == bpsk.demodulate(y, np.exp(1j * 0.4))).all()
# 16-QAM: unit energy; its bit error rate over AWGN is about (3/4) Q(sqrt(SNR / 5)) (Gray labelling)
qam = SionnaModulation('qam', 4)
b = rng.integers(0, 2, 400_000)
x = qam.modulate(b)
assert qam.bits_per_symbol == 4 and x.shape == (100_000,) and abs((np.abs(x) ** 2).mean() - 1) < 0.01
h = 0.5 * np.exp(1j * 1.1)
for snr_db in (8, 12):
    noise_power = abs(h) ** 2 * 10 ** (-snr_db / 10)
    errors = (qam.demodulate(apply_channel(x, h, 1.0, noise_power, rng), h) != b).mean()
    expected = 0.75 * 0.5 * erfc(sqrt(10 ** (snr_db / 10) / 5) / sqrt(2))
    assert abs(errors - expected) < 0.15 * expected, (snr_db, errors, expected)
# torch tensors stay torch tensors
assert torch.is_tensor(qam.modulate(torch.as_tensor(b[:8]))) and torch.is_tensor(qam.demodulate(torch.as_tensor(x[:2]), torch.tensor(1.0)))

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()